In [1]:
import pandas as pd

ames = pd.read_csv("./data/ames_prices.csv")
ames["central_air"] = (ames["Central.Air"] == "Y").astype(int)

p_hat = ames["central_air"].mean()
print(f"Estimated proportion: {p_hat:.4f}")

Estimated proportion: 0.9331


In [3]:
import numpy as np

seed = 40
B = 2000
rng = np.random.default_rng(seed)

bootstrap_props = np.array([
    rng.choice(ames["central_air"], size=len(ames), replace=True).mean()
    for _ in range(B)
])

print(f"Seed: {seed}; Bootstrap replications: {B}")

Seed: 40; Bootstrap replications: 2000


In [4]:
bootstrap_se = bootstrap_props.std(ddof=1)
print(f"Bootstrap standard error: {bootstrap_se:.4f}")

Bootstrap standard error: 0.0048


In [6]:
for confidence, z in [(90, 1.645), (95, 1.96), (99, 2.576)]:
    lower = p_hat - z * bootstrap_se
    upper = p_hat + z * bootstrap_se
    print(f"{confidence}% confidence interval: ({lower:.4f}, {upper:.4f})")

90% confidence interval: (0.9252, 0.9410)
95% confidence interval: (0.9237, 0.9425)
99% confidence interval: (0.9207, 0.9455)


In [9]:
x = np.arcsinh(ames["area"].to_numpy())
y = np.arcsinh(ames["price"].to_numpy())

r_hat = np.corrcoef(x, y)[0, 1]

seed = 40
B = 2000
rng = np.random.default_rng(seed)

bootstrap_corrs = np.empty(B)
for b in range(B):
    indices = rng.choice(len(ames), size=len(ames), replace=True)
    bootstrap_corrs[b] = np.corrcoef(x[indices], y[indices])[0, 1]

print(f"Estimated Pearson correlation: {r_hat:.4f}")
print(f"Seed: {seed}; Bootstrap replications: {B}")

Estimated Pearson correlation: 0.7232
Seed: 40; Bootstrap replications: 2000


In [10]:
bootstrap_corr_se = bootstrap_corrs.std(ddof=1)
print(f"Bootstrap standard error: {bootstrap_corr_se:.4f}")

Bootstrap standard error: 0.0100


In [11]:
for confidence, z in [(90, 1.645), (95, 1.96), (99, 2.576)]:
    lower = r_hat - z * bootstrap_corr_se
    upper = r_hat + z * bootstrap_corr_se
    print(f"{confidence}% confidence interval: ({lower:.4f}, {upper:.4f})")

90% confidence interval: (0.7067, 0.7397)
95% confidence interval: (0.7035, 0.7428)
99% confidence interval: (0.6973, 0.7490)


In [12]:
metabric = pd.read_csv("./data/metabric.csv")
times = metabric["Overall Survival (Months)"].dropna().to_numpy()

mean_hat = times.mean()

seed = 42
B = 2000
rng = np.random.default_rng(seed)

bootstrap_means = np.array([
    rng.choice(times, size=len(times), replace=True).mean()
    for _ in range(B)
])

print(f"Estimated mean: {mean_hat:.2f} months")
print(f"Seed: {seed}; Bootstrap replications: {B}")

Estimated mean: 129.09 months
Seed: 42; Bootstrap replications: 2000


In [13]:
bootstrap_mean_se = bootstrap_means.std(ddof=1)
print(f"Bootstrap standard error: {bootstrap_mean_se:.4f} months")

Bootstrap standard error: 2.1358 months


In [14]:
for confidence, z in [(90, 1.645), (95, 1.96), (99, 2.576)]:
    lower = mean_hat - z * bootstrap_mean_se
    upper = mean_hat + z * bootstrap_mean_se
    decision = "Reject H0" if not lower <= 120 <= upper else "Fail to reject H0"

    print(
        f"{confidence}% CI: ({lower:.2f}, {upper:.2f}) months; "
        f"({lower / 12:.2f}, {upper / 12:.2f}) years. "
        f"At alpha = {1 - confidence / 100:.2f}: {decision}"
    )

90% CI: (125.58, 132.61) months; (10.47, 11.05) years. At alpha = 0.10: Reject H0
95% CI: (124.91, 133.28) months; (10.41, 11.11) years. At alpha = 0.05: Reject H0
99% CI: (123.59, 134.60) months; (10.30, 11.22) years. At alpha = 0.01: Reject H0


In [15]:
nhanes = pd.read_csv("./data/nhanes_data_17_18.csv", low_memory=False)

sleep = nhanes.loc[
    nhanes["AgeInYearsAtScreening"] >= 18,
    "SleepHoursWeekdaysOrWorkdays"
].dropna().to_numpy()

sleep_mean = sleep.mean()

seed = 42
B = 2000
rng = np.random.default_rng(seed)

bootstrap_sleep_means = np.array([
    rng.choice(sleep, size=len(sleep), replace=True).mean()
    for _ in range(B)
])

print(f"Respondents: {len(sleep)}")
print(f"Estimated mean: {sleep_mean:.4f} hours")
print(f"Seed: {seed}; Bootstrap replications: {B}")

Respondents: 5489
Estimated mean: 7.6212 hours
Seed: 42; Bootstrap replications: 2000


In [16]:
bootstrap_sleep_se = bootstrap_sleep_means.std(ddof=1)
print(f"Bootstrap standard error: {bootstrap_sleep_se:.4f} hours")

Bootstrap standard error: 0.0228 hours


In [17]:
for confidence, z in [(90, 1.645), (95, 1.96), (99, 2.576)]:
    lower = sleep_mean - z * bootstrap_sleep_se
    upper = sleep_mean + z * bootstrap_sleep_se
    decision = "Reject H0" if not lower <= 8 <= upper else "Fail to reject H0"

    print(
        f"{confidence}% CI: ({lower:.4f}, {upper:.4f}) hours. "
        f"At alpha = {1 - confidence / 100:.2f}: {decision}"
    )

90% CI: (7.5838, 7.6587) hours. At alpha = 0.10: Reject H0
95% CI: (7.5767, 7.6658) hours. At alpha = 0.05: Reject H0
99% CI: (7.5626, 7.6798) hours. At alpha = 0.01: Reject H0
